# Intelligent AI Assistant for Company Documents

## Retrieval-Augmented Generation (RAG)

### Project Overview

This project implements an intelligent AI assistant capable of answering questions about a collection of company documents using **Retrieval-Augmented Generation (RAG)**.

Unlike a traditional chatbot that relies only on the knowledge stored in a language model, this assistant retrieves relevant information from a document collection before generating an answer.

The system is designed to be **dataset-independent**: the underlying documents can be replaced without modifying the core RAG architecture.

### Main Objective

The objective is to build a reusable pipeline capable of:

1. Loading PDF documents.
2. Extracting their textual content.
3. Cleaning and preprocessing the extracted text.
4. Splitting documents into searchable chunks.
5. Converting chunks into semantic embeddings.
6. Storing and searching those embeddings.
7. Retrieving the most relevant information for a user question.
8. Providing the retrieved information to a language model.
9. Generating a grounded answer based on the available documents.
10. Providing the sources used to construct the answer.

### High-Level Architecture

**PDF Documents → Text Extraction → Cleaning → Chunking → Embeddings → Vector Store → Retrieval → LLM → Answer + Sources**

### Important Design Principle

The assistant should not depend on a specific company, document collection, or domain.

The document collection is treated as an external knowledge base. Therefore, replacing the contents of the document directory should allow the same system to operate on a different dataset after rebuilding the document index.


## 1. Environment Setup

Before implementing the RAG pipeline, we install the libraries required for document processing, semantic search, and language-model interaction.

The main components are:

* **PyPDF** — extraction of text from PDF documents.
* **Sentence Transformers** — generation of semantic embeddings.
* **NumPy** — numerical operations and similarity calculations.
* **Transformers** — interaction with language models.
* **Gradio** — optional user interface for interacting with the assistant.

The dependencies are installed once and then imported throughout the notebook.


In [43]:
%pip install -q pypdf sentence-transformers chromadb numpy transformers accelerate gradio reportlab rank-bm25

Note: you may need to restart the kernel to use updated packages.


## 1. Project Configuration

In this section, we define the paths and configuration used throughout
the notebook.

The document directory is intentionally separated from the processing
logic. This allows the dataset to be replaced later without modifying
the RAG pipeline.

In [1]:
from pathlib import Path
import re
from collections import Counter

import numpy as np
import torch

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "documents"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"

DATA_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT.resolve())
print("Documents    :", DATA_DIR.resolve())
print("ChromaDB     :", CHROMA_DIR.resolve())

Project root : C:\Users\PC\CERIST\intelligent-rag-assistant
Documents    : C:\Users\PC\CERIST\intelligent-rag-assistant\data\documents
ChromaDB     : C:\Users\PC\CERIST\intelligent-rag-assistant\data\chroma


## 2. Dataset Discovery

The assistant should automatically detect the available PDF reports.

At this stage, the real company documents may not yet be available.
This is not a problem: we can build and test the complete pipeline
using temporary demonstration documents.

When the real reports become available, they can simply be placed
inside `data/documents/`.

In [2]:
pdf_files = sorted(DATA_DIR.glob("*.pdf"))

print(f"Number of PDF documents found: {len(pdf_files)}")

for pdf_file in pdf_files:
    print(f" - {pdf_file.name}")

Number of PDF documents found: 3
 - company_overview.pdf
 - financial_report_2025.pdf
 - operations_report_2025.pdf


## 3. PDF Text Extraction

PDF documents are not directly usable by the language model.

We first extract their textual content.

The extraction process preserves page-level information because page
numbers will later be used for source attribution.

Each extracted page becomes a structured record:

{
    "source": "financial_report_2025.pdf",
    "page": 1,
    "text": "..."
}

Keeping this metadata is important for traceability.

In [3]:
from pypdf import PdfReader

In [4]:
def extract_pdf_text(pdf_path):
    """
    Extract text from all pages of a PDF.

    Parameters
    ----------
    pdf_path : Path
        Path to the PDF document.

    Returns
    -------
    list[dict]
        One dictionary per page containing the page text
        and source metadata.
    """
    
    reader = PdfReader(pdf_path)
    
    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        
        pages.append({
            "source": pdf_path.name,
            "page": page_number,
            "text": text
        })

    return pages

In [5]:
documents = []

for pdf_path in pdf_files:
    pages = extract_pdf_text(pdf_path)
    documents.extend(pages)

print(f"✓ Loaded {len(pdf_files)} PDF(s)")
print(f"✓ Extracted {len(documents)} page(s)")

✓ Loaded 3 PDF(s)
✓ Extracted 3 page(s)


In [6]:
documents[0]

{'source': 'company_overview.pdf',
 'page': 1,
 'text': 'Example Company - Company Overview\nCompany Activities\nExample Company develops software solutions for businesses. Its main activities include backend\ndevelopment, cloud services, data management, cybersecurity and artificial intelligence.\nEmployees\nThe company has 120 employees distributed across engineering, operations, sales, finance and\nmanagement departments.\nHeadquarters\nThe company headquarters are located in Algiers.\n'}

## 4. Text Cleaning

PDF extraction can introduce unnecessary whitespace, line breaks and
other formatting artifacts.

We normalize the extracted text before creating chunks.

The cleaning process should remain conservative: we do not want to
modify the actual meaning of the document.

In [7]:
def clean_text(text):
    """
    Basic PDF text normalization.
    """
    
    # Replace repeated whitespace
    text = re.sub(r"\s+", " ", text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    return text

In [8]:
for document in documents:
    document["text"] = clean_text(document["text"])

In [9]:
documents = [
    document
    for document in documents
    if document["text"]
]

print(f"✓ {len(documents)} non-empty pages remain.")

✓ 3 non-empty pages remain.


## 5. Document Chunking

Large documents should not be sent directly to the language model.

Instead, each page is divided into smaller chunks.

Chunking improves retrieval because the embedding represents a focused
piece of information rather than an entire large document.

We use overlapping chunks.

Example:

Chunk 1:
AAAA BBBB CCCC DDDD

Chunk 2:
        CCCC DDDD EEEE FFFF

The overlap helps preserve context between neighboring chunks.

In [10]:
def chunk_text(text, chunk_size=800, overlap=120):
    """
    Split text into overlapping chunks.

    Parameters
    ----------
    text : str
        Text to split.
    chunk_size : int
        Approximate maximum chunk size in characters.
    overlap : int
        Number of characters shared between consecutive chunks.

    Returns
    -------
    list[str]
        Generated text chunks.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than 0")

    if overlap < 0:
        raise ValueError("overlap cannot be negative")

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []

    start = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks

## 6. Chunk Metadata

Each chunk keeps information about where it came from.

This allows the assistant to provide source attribution such as:

`financial_report_2025.pdf — page 1`

instead of returning an answer with no indication of its origin.

In [12]:
chunks = []

for document in documents:

    page_chunks = chunk_text(
        document["text"],
        chunk_size=800,
        overlap=120
    )

    for chunk_number, chunk in enumerate(page_chunks):

        chunks.append({
            "chunk_id": (
                f"{document['source']}"
                f"_page_{document['page']}"
                f"_chunk_{chunk_number}"
            ),
            "source": document["source"],
            "page": document["page"],
            "chunk_number": chunk_number,
            "text": chunk
        })

In [13]:
print(f"Total chunks: {len(chunks)}")

Total chunks: 3


In [13]:
chunks[0]

{'chunk_id': 'company_overview.pdf_page_1_chunk_0',
 'source': 'company_overview.pdf',
 'page': 1,
 'chunk_number': 0,
 'text': 'Example Company - Company Overview Company Activities Example Company develops software solutions for businesses. Its main activities include backend development, cloud services, data management, cybersecurity and artificial intelligence. Employees The company has 120 employees distributed across engineering, operations, sales, finance and management departments. Headquarters The company headquarters are located in Algiers.'}

In [14]:
for i, chunk in enumerate(chunks[:5]):

    print("=" * 80)
    print(f"Chunk {i}")
    print(f"Source : {chunk['source']}")
    print(f"Page   : {chunk['page']}")
    print(f"ID     : {chunk['chunk_id']}")
    print()
    print(chunk["text"])

Chunk 0
Source : company_overview.pdf
Page   : 1
ID     : company_overview.pdf_page_1_chunk_0

Example Company - Company Overview Company Activities Example Company develops software solutions for businesses. Its main activities include backend development, cloud services, data management, cybersecurity and artificial intelligence. Employees The company has 120 employees distributed across engineering, operations, sales, finance and management departments. Headquarters The company headquarters are located in Algiers.
Chunk 1
Source : financial_report_2025.pdf
Page   : 1
ID     : financial_report_2025.pdf_page_1_chunk_0

Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.
Chunk 2
Source : operations_report_2025.pdf

In [16]:
from rank_bm25 import BM25Okapi
def tokenize(text):
    """
    Basic tokenizer for BM25 keyword search.
    """
    return re.findall(r"\b\w+\b", text.lower())


tokenized_chunks = [
    tokenize(chunk["text"])
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index created for {len(chunks)} chunks.")


BM25 index created for 3 chunks.


In [ ]:
def retrieve_bm25(query, k=5):
    """
    Retrieve chunks using BM25 keyword matching.
    """

    if not query.strip():
        raise ValueError("Query cannot be empty.")

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:k]

    results = []

    for index in top_indices:

        results.append({
            "chunk_id": chunks[index]["chunk_id"],
            "text": chunks[index]["text"],
            "source": chunks[index]["source"],
            "page": chunks[index]["page"],
            "chunk_number": chunks[index]["chunk_number"],
            "score": float(scores[index])
        })

    return results

In [18]:
results = retrieve_bm25(
    "What was the company's revenue in 2025?",
    k=5
)

for rank, result in enumerate(results, start=1):

    print("=" * 80)
    print(f"Rank   : {rank}")
    print(f"Score  : {result['score']:.4f}")
    print(f"Source : {result['source']}")
    print(f"Page   : {result['page']}")
    print()
    print(result["text"])

Rank   : 1
Score  : 1.0675
Source : financial_report_2025.pdf
Page   : 1

Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.
Rank   : 2
Score  : 0.2289
Source : company_overview.pdf
Page   : 1

Example Company - Company Overview Company Activities Example Company develops software solutions for businesses. Its main activities include backend development, cloud services, data management, cybersecurity and artificial intelligence. Employees The company has 120 employees distributed across engineering, operations, sales, finance and management departments. Headquarters The company headquarters are located in Algiers.
Rank   : 3
Score  : 0.2239
Source : operations_report_2025.pdf
Page   : 1

Example Company - Operati

## 7. Dataset Statistics

Before building the retrieval system, we inspect basic statistics
about the processed dataset.

These measurements help us detect problems such as:

- empty documents
- unusually short documents
- unexpectedly large documents
- an excessive number of chunks

In [22]:
from collections import Counter

source_counts = Counter(
    chunk["source"]
    for chunk in chunks
)

print("Documents represented in chunks:")
print()

for source, count in source_counts.items():
    print(f"{source}: {count} chunks")

Documents represented in chunks:

company_overview.pdf: 1 chunks
financial_report_2025.pdf: 1 chunks
operations_report_2025.pdf: 1 chunks


In [23]:
chunk_lengths = [
    len(chunk["text"])
    for chunk in chunks
]

print()
print("Chunk statistics")
print("----------------")
print(f"Number of chunks : {len(chunk_lengths)}")
print(f"Minimum length   : {min(chunk_lengths)}")
print(f"Maximum length   : {max(chunk_lengths)}")
print(f"Average length   : {np.mean(chunk_lengths):.1f}")


Chunk statistics
----------------
Number of chunks : 3
Minimum length   : 328
Maximum length   : 444
Average length   : 399.7


## 8. Semantic Embeddings

Keyword search looks for exact words.

Semantic search instead represents text as numerical vectors called
**embeddings**.

Texts with similar meanings should have vectors that are close to
each other in the embedding space.

For example:

"How much money did the company make?"

and

"What was the company's revenue?"

may use different words but express a similar concept.

An embedding model converts both sentences into numerical vectors
that can be compared mathematically.

In [24]:
from sentence_transformers import SentenceTransformer

c:\Users\PC\CERIST\intelligent-rag-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model:", EMBEDDING_MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3139.38it/s]


Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [26]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", embeddings.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00, 12.81it/s]

Embedding matrix shape: (3, 384)


## 11. ChromaDB Vector Store

ChromaDB is a vector database designed for storing and retrieving
embedding-based information.

Instead of manually maintaining:

- vectors
- document texts
- metadata
- similarity calculations

we can store these elements together in a ChromaDB collection.

Each chunk contains:

- a unique ID
- its text
- its embedding
- metadata such as source and page number

ChromaDB can also persist the collection on disk.

This makes it more appropriate for a reusable RAG application than
keeping embeddings only in memory.

In [27]:
import chromadb

In [28]:
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

collection = chroma_client.get_or_create_collection(
    name="company_documents"
)

print("Collection:", collection.name)

Collection: company_documents


In [29]:
existing_count = collection.count()

print("Existing chunks in ChromaDB:", existing_count)

Existing chunks in ChromaDB: 3


In [30]:
chroma_client.delete_collection("company_documents")

collection = chroma_client.get_or_create_collection(
    name="company_documents"
)

print("Collection reset.")

Collection reset.


In [31]:
collection.add(
    ids=[
        chunk["chunk_id"]
        for chunk in chunks
    ],
    documents=[
        chunk["text"]
        for chunk in chunks
    ],
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "source": chunk["source"],
            "page": chunk["page"],
            "chunk_number": chunk["chunk_number"]
        }
        for chunk in chunks
    ]
)

print("Chunks stored in ChromaDB:", collection.count())

Chunks stored in ChromaDB: 3


## 12. Semantic Retrieval with ChromaDB

When a user asks a question:

1. The question is converted into an embedding.
2. ChromaDB searches for similar vectors.
3. The closest chunks are returned.
4. Their metadata is preserved.

The distance returned by ChromaDB is converted into a similarity score
for easier interpretation.

Because our embeddings are normalized, cosine similarity can be
calculated using the dot product.

In [32]:
def retrieve_from_chroma(query, k=5):
    """
    Retrieve candidate chunks from ChromaDB.
    """

    if not query.strip():
        raise ValueError("Query cannot be empty.")

    if k <= 0:
        raise ValueError("k must be greater than 0.")

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=k
    )

    retrieved = []

    for i in range(len(results["ids"][0])):

        distance = results["distances"][0][i]

        # Chroma's default distance for cosine space:
        similarity = 1 - distance

        retrieved.append({
            "chunk_id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "page": results["metadatas"][0][i]["page"],
            "chunk_number": results["metadatas"][0][i]["chunk_number"],
            "score": float(similarity)
        })

    return retrieved

In [33]:
results = retrieve_from_chroma(
    "What was the company's revenue in 2025?",
    k=5
)

for rank, result in enumerate(results, start=1):

    print("=" * 80)
    print(f"Rank   : {rank}")
    print(f"Score  : {result['score']:.4f}")
    print(f"Source : {result['source']}")
    print(f"Page   : {result['page']}")
    print()
    print(result["text"])

Rank   : 1
Score  : 0.4533
Source : financial_report_2025.pdf
Page   : 1

Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.
Rank   : 2
Score  : -0.1916
Source : operations_report_2025.pdf
Page   : 1

Example Company - Operations Report 2025 Technology Department The technology department manages backend systems, databases, cloud infrastructure and internal software platforms. Cybersecurity The company uses access control, authentication mechanisms, network monitoring and regular security assessments. Artificial Intelligence The company is developing artificial intelligence solutions for document analysis and business process automation.
Rank   : 3
Score  : -0.2542
Source : company_overview.pdf
Page   : 1

Exampl

## 15. Hybrid Search

The system now combines two complementary retrieval strategies:

### Semantic Search

Semantic search uses embeddings to identify chunks with similar meaning
to the user's question.

It is useful when the question and document use different wording.

### Keyword Search

Keyword search uses BM25 to identify chunks containing important terms
from the user's question.

It is particularly useful for exact terms such as:

- names
- years
- technical terms
- identifiers
- specific expressions

### Hybrid Search

The two retrieval methods are combined using Reciprocal Rank Fusion (RRF).

RRF combines the rankings produced by the individual retrieval systems
without requiring their scores to be on the same scale.

The resulting ranking benefits from both semantic similarity and
lexical matching.

In [34]:
from collections import defaultdict

In [35]:
RRF_K = 60


def reciprocal_rank_fusion(
    semantic_results,
    keyword_results,
    k=RRF_K
):
    """
    Combine semantic and keyword retrieval results
    using Reciprocal Rank Fusion (RRF).
    """

    fused_scores = defaultdict(float)
    result_data = {}

    # -----------------------------
    # Semantic ranking
    # -----------------------------

    for rank, result in enumerate(
        semantic_results,
        start=1
    ):

        chunk_id = result["chunk_id"]

        fused_scores[chunk_id] += (
            1 / (k + rank)
        )

        if chunk_id not in result_data:
            result_data[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result["text"],
                "source": result["source"],
                "page": result["page"],
                "chunk_number": result["chunk_number"],
            }

        result_data[chunk_id]["semantic_score"] = (
            result["score"]
        )

    # -----------------------------
    # Keyword ranking
    # -----------------------------

    for rank, result in enumerate(
        keyword_results,
        start=1
    ):

        chunk_id = result["chunk_id"]

        fused_scores[chunk_id] += (
            1 / (k + rank)
        )

        if chunk_id not in result_data:
            result_data[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result["text"],
                "source": result["source"],
                "page": result["page"],
                "chunk_number": result["chunk_number"],
            }

        result_data[chunk_id]["bm25_score"] = (
            result["score"]
        )

    # -----------------------------
    # Final ranking
    # -----------------------------

    ranked_chunks = sorted(
        fused_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    hybrid_results = []

    for chunk_id, rrf_score in ranked_chunks:

        result = result_data[chunk_id].copy()

        result["rrf_score"] = rrf_score

        hybrid_results.append(result)

    return hybrid_results

In [36]:
query = "What was the company's revenue in 2025?"

semantic_results = retrieve_from_chroma(
    query,
    k=5
)

keyword_results = retrieve_bm25(
    query,
    k=5
)

hybrid_results = reciprocal_rank_fusion(
    semantic_results,
    keyword_results
)

In [37]:
for rank, result in enumerate(
    hybrid_results[:5],
    start=1
):

    print("=" * 80)

    print(f"Rank          : {rank}")
    print(f"RRF score     : {result['rrf_score']:.6f}")

    print(
        f"Semantic score: "
        f"{result.get('semantic_score', 0):.4f}"
    )

    print(
        f"BM25 score    : "
        f"{result.get('bm25_score', 0):.4f}"
    )

    print(f"Source        : {result['source']}")
    print(f"Page          : {result['page']}")
    print()

    print(result["text"])

Rank          : 1
RRF score     : 0.032787
Semantic score: 0.4533
BM25 score    : 1.0675
Source        : financial_report_2025.pdf
Page          : 1

Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.
Rank          : 2
RRF score     : 0.032002
Semantic score: -0.1916
BM25 score    : 0.2239
Source        : operations_report_2025.pdf
Page          : 1

Example Company - Operations Report 2025 Technology Department The technology department manages backend systems, databases, cloud infrastructure and internal software platforms. Cybersecurity The company uses access control, authentication mechanisms, network monitoring and regular security assessments. Artificial Intelligence The company is developing artificial in

## 13. Relevance Filtering

A vector database will always try to return the closest chunks.

However, the closest chunks are not necessarily relevant enough to
answer the question.

For example:

"What is the company's office in Tokyo?"

may return company information even though none of the documents mention
Tokyo.

Therefore, retrieval has two stages:

1. Candidate retrieval
2. Relevance filtering

Only chunks whose similarity score passes the threshold are considered
relevant.

The threshold should be tuned using evaluation data rather than treated
as a universal value.

In [38]:
def retrieve_hybrid(
    query,
    k=5,
    semantic_k=5,
    keyword_k=5
):
    """
    Hybrid retrieval combining:

    - semantic search through ChromaDB
    - keyword search through BM25
    - Reciprocal Rank Fusion
    """

    semantic_results = retrieve_from_chroma(
        query,
        k=semantic_k
    )

    keyword_results = retrieve_bm25(
        query,
        k=keyword_k
    )

    hybrid_results = reciprocal_rank_fusion(
        semantic_results,
        keyword_results
    )

    return hybrid_results[:k]

In [39]:
query = "How much did the company invest in AI and cybersecurity in 2025?"

results = retrieve_hybrid(
    query,
    k=5
)

for rank, result in enumerate(results, start=1):

    print("=" * 80)

    print(f"Rank      : {rank}")
    print(f"RRF score : {result['rrf_score']:.6f}")
    print(f"Source    : {result['source']}")
    print(f"Page      : {result['page']}")
    print()

    print(result["text"])

Rank      : 1
RRF score : 0.032787
Source    : financial_report_2025.pdf
Page      : 1

Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.
Rank      : 2
RRF score : 0.032002
Source    : operations_report_2025.pdf
Page      : 1

Example Company - Operations Report 2025 Technology Department The technology department manages backend systems, databases, cloud infrastructure and internal software platforms. Cybersecurity The company uses access control, authentication mechanisms, network monitoring and regular security assessments. Artificial Intelligence The company is developing artificial intelligence solutions for document analysis and business process automation.
Rank      : 3
RRF score : 0.032002
Source    : co

In [40]:
query = "What cybersecurity measures does the company use?"

results = retrieve_hybrid(
    query,
    k=5
)

for rank, result in enumerate(results, start=1):

    print("=" * 80)

    print(f"Rank      : {rank}")
    print(f"RRF score : {result['rrf_score']:.6f}")
    print(f"Source    : {result['source']}")
    print(f"Page      : {result['page']}")
    print()

    print(result["text"])

Rank      : 1
RRF score : 0.032522
Source    : operations_report_2025.pdf
Page      : 1

Example Company - Operations Report 2025 Technology Department The technology department manages backend systems, databases, cloud infrastructure and internal software platforms. Cybersecurity The company uses access control, authentication mechanisms, network monitoring and regular security assessments. Artificial Intelligence The company is developing artificial intelligence solutions for document analysis and business process automation.
Rank      : 2
RRF score : 0.032266
Source    : company_overview.pdf
Page      : 1

Example Company - Company Overview Company Activities Example Company develops software solutions for businesses. Its main activities include backend development, cloud services, data management, cybersecurity and artificial intelligence. Employees The company has 120 employees distributed across engineering, operations, sales, finance and management departments. Headquarters The 

In [41]:
def retrieve_hybrid(
    query,
    k=5,
    semantic_k=5,
    keyword_k=5
):
    """
    Hybrid retrieval combining:

    1. Semantic search through ChromaDB
    2. Keyword search through BM25
    3. Reciprocal Rank Fusion (RRF)
    """

    if not query.strip():
        raise ValueError("Query cannot be empty.")

    if k <= 0:
        raise ValueError("k must be greater than 0.")

    if semantic_k <= 0:
        raise ValueError(
            "semantic_k must be greater than 0."
        )

    if keyword_k <= 0:
        raise ValueError(
            "keyword_k must be greater than 0."
        )

    # -----------------------------
    # 1. Semantic retrieval
    # -----------------------------
    semantic_results = retrieve_from_chroma(
        query,
        k=semantic_k
    )

    # -----------------------------
    # 2. Keyword retrieval
    # -----------------------------
    keyword_results = retrieve_bm25(
        query,
        k=keyword_k
    )

    # -----------------------------
    # 3. Fuse the rankings
    # -----------------------------
    hybrid_results = reciprocal_rank_fusion(
        semantic_results,
        keyword_results
    )

    # -----------------------------
    # 4. Return final top-k
    # -----------------------------
    return hybrid_results[:k]

In [43]:
def display_retrieval_results(
    results,
    title
):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    for rank, result in enumerate(
        results,
        start=1
    ):
        print(
            f"{rank}. "
            f"{result['source']} "
            f"(page {result['page']})"
        )

        if "rrf_score" in result:
            print(
                f"   RRF score: "
                f"{result['rrf_score']:.6f}"
            )

        if "semantic_score" in result:
            print(
                f"   Semantic: "
                f"{result['semantic_score']:.4f}"
            )

        if "bm25_score" in result:
            print(
                f"   BM25: "
                f"{result['bm25_score']:.4f}"
            )

        print(
            f"   {result['text'][:300]}..."
        )

In [55]:
def unique_sources(results):
    """
    Extract unique document sources from retrieved results.

    For hybrid retrieval, the RRF score is used as the
    ranking score.
    """

    seen = set()
    sources = []

    for result in results:
        key = (
            result["source"],
            result["page"]
        )

        if key not in seen:
            seen.add(key)

            sources.append({
                "source": result["source"],
                "page": result["page"],
                "score": result.get(
                    "rrf_score",
                    0.0
                )
            })

    return sources

## 15. Context Construction

The retrieved chunks are converted into a structured context that will
be provided to the language model.

Each piece of context contains its source and page number.

This allows the model to understand where the information came from.

In [45]:
def build_context(results):
    """
    Convert retrieved chunks into a formatted context string.
    """

    context_parts = []

    for i, result in enumerate(results, start=1):

        context_parts.append(
            f"[Document {i}]\n"
            f"Source: {result['source']}\n"
            f"Page: {result['page']}\n"
            f"Content:\n{result['text']}"
        )

    return "\n\n".join(context_parts)

## 16. Language Model

The retriever finds relevant information, but it does not generate
natural-language answers.

We therefore use a local Qwen instruction-following model.

The model size is selected according to the available hardware:

- GPU → Qwen 2.5 1.5B
- CPU → Qwen 2.5 0.5B

The model is used only as the generation component.

Company-specific information should come from the retrieved documents.

In [46]:
from transformers import pipeline

In [47]:
DEVICE = 0 if torch.cuda.is_available() else -1

print(
    "CUDA available:",
    torch.cuda.is_available()
)

print(
    "Device:",
    "GPU" if DEVICE == 0 else "CPU"
)

CUDA available: False
Device: CPU


In [48]:
if DEVICE == 0:
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
else:
    MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Selected model:", MODEL_NAME)

Selected model: Qwen/Qwen2.5-0.5B-Instruct


In [49]:
generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device=DEVICE
)

print("Language model loaded.")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 340.66it/s]


Language model loaded.


## 17. Prompt Engineering

The language model receives:

- the user's question
- the retrieved document context
- explicit instructions about how to use the context

The assistant must not invent company-specific information.

If the retrieved documents do not contain enough information, the
assistant should explicitly say that it does not know based on the
provided documents.

In [50]:
SYSTEM_PROMPT = """
You are an AI assistant for a company.

Your task is to answer questions using only the information
provided in the retrieved company documents.

Rules:

1. Use only the supplied documents as factual sources.
2. Do not invent information.
3. Do not use outside knowledge for company-specific questions.
4. If the documents do not contain enough information, say:
   "I don't know based on the provided documents."
5. Keep the answer concise and clear.
6. Do not mention information that is not supported by the context.
"""

In [51]:
def build_rag_prompt(question, context):
    """
    Build the prompt sent to the language model.
    """

    return f"""
{SYSTEM_PROMPT}

Retrieved documents:

{context}

User question:

{question}

Answer using only the retrieved documents.
"""

## 18. Clean LLM Generation

The text-generation pipeline can return the input prompt together with
the generated text.

For a user-facing application, we only want the generated answer.

Therefore, `return_full_text=False` is used.

In [52]:
def generate_answer(prompt, max_new_tokens=250):
    """
    Generate only the newly generated answer text.
    """

    response = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False
    )

    return response[0]["generated_text"].strip()

## 19. Complete RAG Pipeline

The complete pipeline now combines all previous components.

Question
    ↓
ChromaDB Retrieval
    ↓
Relevance Filtering
    ↓
Context Construction
    ↓
Prompt Construction
    ↓
Qwen Generation
    ↓
Answer + Sources

The pipeline also handles questions for which no sufficiently relevant
document chunks were found.

In [53]:
def answer_question(
    question,
    k=5
):
    """
    Complete RAG pipeline using hybrid retrieval.
    """

    if not question.strip():
        raise ValueError(
            "Question cannot be empty."
        )

    # -----------------------------
    # 1. Hybrid retrieval
    # -----------------------------
    results = retrieve_hybrid(
        question,
        k=k,
        semantic_k=8,
        keyword_k=8
    )

    # -----------------------------
    # 2. No retrieved documents
    # -----------------------------
    if not results:
        return {
            "question": question,
            "answer": (
                "I don't know based on "
                "the provided documents."
            ),
            "sources": [],
            "retrieved_chunks": []
        }

    # -----------------------------
    # 3. Build context
    # -----------------------------
    context = build_context(
        results
    )

    # -----------------------------
    # 4. Build RAG prompt
    # -----------------------------
    prompt = build_rag_prompt(
        question,
        context
    )

    # -----------------------------
    # 5. Generate answer
    # -----------------------------
    answer = generate_answer(
        prompt
    )

    # -----------------------------
    # 6. Collect sources
    # -----------------------------
    sources = unique_sources(
        results
    )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": results
    }

In [56]:
result = answer_question(
    "What was the company's revenue in 2025?"
)

print("ANSWER")
print("------")
print(result["answer"])

print()
print("SOURCES")
print("-------")

for source in result["sources"]:
    print(
        f"- {source['source']} "
        f"— page {source['page']}"
    )

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER
------
Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year.

SOURCES
-------
- financial_report_2025.pdf — page 1
- operations_report_2025.pdf — page 1
- company_overview.pdf — page 1


## 22. Interactive Assistant

After validating the RAG pipeline programmatically, we can expose it
through a simple web interface using Gradio.

The interface allows a user to:

1. Enter a question.
2. Retrieve relevant company information.
3. Generate an answer.
4. Display the supporting sources.

In [59]:
import gradio as gr

In [60]:
def chat_with_documents(question):
    """
    Gradio interface function.
    """

    result = answer_question(question)

    answer = result["answer"]

    if result["sources"]:

        source_text = "\n\n### Sources\n"

        for source in result["sources"]:
            source_text += (
                f"- {source['source']} "
                f"— page {source['page']}\n"
            )

        answer += source_text

    return answer

In [61]:
demo = gr.Interface(
    fn=chat_with_documents,
    inputs=gr.Textbox(
        label="Ask a question",
        placeholder="Ask something about the company reports..."
    ),
    outputs=gr.Markdown(
        label="Answer"
    ),
    title="Company Reports AI Assistant",
    description=(
        "Ask questions about the documents available "
        "in the knowledge base."
    )
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

## 23. Security Considerations

A company RAG system may process sensitive internal information.

Important security considerations include:

### Document security

Only authorized documents should be placed in the knowledge base.

### Access control

Users should only retrieve documents they are authorized to access.

### Prompt injection

Retrieved documents should be treated as untrusted content.

A malicious document could contain instructions attempting to manipulate
the language model.

The system should distinguish between:

- instructions from the application
- user questions
- retrieved document content

### Data privacy

Sensitive company information should not be sent to external APIs
without authorization.

Using a local embedding model and local language model can reduce
external data exposure.

### Logging

Production systems should avoid logging sensitive document content
unnecessarily.

### File validation

Uploaded documents should be validated before processing.

### Authentication

The final application should require authentication if it contains
internal company information.

## 24. Current Limitations

This implementation is a development prototype.

Several improvements may be required for production use.

### PDF extraction

Some PDFs contain scanned images rather than selectable text.

OCR would be required for these documents.

### Tables

Financial and technical reports often contain tables.

Basic text extraction may not preserve their structure correctly.

A specialized table extraction strategy may therefore be required.

### Chunking

The current chunking strategy uses character-based chunks.

More advanced approaches could use:

- sentence-based chunking
- paragraph-based chunking
- semantic chunking
- document-specific chunking

### Retrieval

The current system uses dense semantic retrieval.

Hybrid retrieval could combine:

- semantic search
- keyword search
- metadata filtering

### Evaluation

The current evaluation dataset is small and fictional.

A real evaluation dataset should be created from representative company
questions.

### Language model

The selected Qwen model is intentionally small for local development.

A larger model may provide better generation quality if sufficient
hardware is available.

## 10. Retrieval Evaluation

Retrieval quality is one of the most important components of a RAG system.

A powerful language model cannot compensate for completely irrelevant
context.

We therefore test the retriever using questions whose answers are known
to exist in our demonstration documents.

For each question, we inspect:

- the retrieved chunks
- their similarity scores
- their source documents
- whether the expected information was retrieved

## 25. Possible Production Architecture

The prototype can later be transformed into a production architecture.

                ┌──────────────────────┐
                │    Company PDFs      │
                └──────────┬───────────┘
                           │
                           ▼
                ┌──────────────────────┐
                │ Document Processing  │
                │ Extraction + OCR     │
                └──────────┬───────────┘
                           │
                           ▼
                ┌──────────────────────┐
                │ Cleaning + Chunking  │
                └──────────┬───────────┘
                           │
                           ▼
                ┌──────────────────────┐
                │ Embedding Model      │
                └──────────┬───────────┘
                           │
                           ▼
                ┌──────────────────────┐
                │      ChromaDB        │
                │    Vector Store      │
                └──────────┬───────────┘
                           │
                    User Question
                           │
                           ▼
                ┌──────────────────────┐
                │ Semantic Retrieval   │
                └──────────┬───────────┘
                           │
                           ▼
                ┌──────────────────────┐
                │ Relevance Filtering  │
                └──────────┬───────────┘
                           │
                           ▼
                ┌──────────────────────┐
                │       LLM            │
                └──────────┬───────────┘
                           │
                           ▼
                ┌──────────────────────┐
                │ Answer + Citations   │
                └──────────────────────┘

A future production implementation could add:

- authentication
- authorization
- document versioning
- OCR
- hybrid search
- reranking
- monitoring
- evaluation pipelines
- API backend
- database-backed user management
- document upload and indexing
- conversation history